# AI-Powered Sustainable Product Recommendation System

This notebook develops an AI recommendation engine that suggests eco-friendly products based on various factors such as user preferences, product category, product similarity, sustainability score, carbon footprint, product ratings, and purchase history. The engine aims to rank products and recommend greener alternatives.

## 1. Install Libraries

In [ ]:
# Install necessary libraries
%pip install pandas numpy scikit-learn sentence-transformers faiss-cpu matplotlib seaborn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 23.9 MB/s eta 0:00:00


## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import faiss
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import random
from typing import List, Dict, Any, Optional

## 3. Generate Dataset

A synthetic dataset will be generated to simulate product information including sustainability metrics, ratings, and basic product details. This dataset will have at least 100 products.

In [ ]:
def generate_sample_dataset(num_products: int = 100) -> pd.DataFrame:
    """
    Generates a sample dataset for sustainable product recommendations.

    Args:
        num_products (int): The number of products to generate.

    Returns:
        pd.DataFrame: A DataFrame containing the sample product data.
    """
    product_ids = [f'P{i:04d}' for i in range(num_products)]
    product_names = [f'Product {i}' for i in range(num_products)]
    brands = ['Brand A', 'Brand B', 'Brand C', 'Brand D', 'Brand E']
    categories = ['Electronics', 'Apparel', 'Home Goods', 'Personal Care', 'Food & Beverage']
    materials = ['Plastic', 'Recycled Plastic', 'Cotton', 'Organic Cotton', 'Glass', 'Metal', 'Bamboo']
    packaging_types = ['Plastic Wrap', 'Cardboard Box', 'Recycled Cardboard', 'No Packaging']

    data = {
        'product_id': product_ids,
        'product_name': product_names,
        'brand': [random.choice(brands) for _ in range(num_products)],
        'category': [random.choice(categories) for _ in range(num_products)],
        'price': np.round(np.random.uniform(5, 500, num_products), 2),
        'rating': np.round(np.random.uniform(2.5, 5.0, num_products), 1),
        'material': [random.choice(materials) for _ in range(num_products)],
        'packaging': [random.choice(packaging_types) for _ in range(num_products)],
        'carbon_footprint': np.round(np.random.uniform(0.1, 10.0, num_products), 2),
        'recyclable': np.random.choice([True, False], num_products, p=[0.7, 0.3]),
        'organic': np.random.choice([True, False], num_products, p=[0.2, 0.8]),
        'eco_certified': np.random.choice([True, False], num_products, p=[0.15, 0.85]),
        'sustainability_score': np.round(np.random.uniform(1, 10, num_products), 1)
    }

    df = pd.DataFrame(data)

    # Simulate some user-product interactions for collaborative filtering later
    user_ids = [f'U{i:03d}' for i in range(20)] # 20 sample users
    interactions_data = []
    for user_id in user_ids:
        num_interactions = random.randint(5, 20) # Each user interacts with 5-20 products
        interacted_product_ids = random.sample(product_ids, num_interactions)
        for prod_id in interacted_product_ids:
            interactions_data.append({'user_id': user_id, 'product_id': prod_id, 'interaction_score': random.randint(1, 5)})

    df_interactions = pd.DataFrame(interactions_data)

    return df, df_interactions

# Generate the dataset
df_products, df_interactions = generate_sample_dataset(num_products=200) # Generating more than 100 for better diversity
print(f"Generated {len(df_products)} product records.")
print(f"Generated {len(df_interactions)} user-product interaction records.")

display(df_products.head())
display(df_interactions.head())

Generated 200 product records.
Generated 262 user-product interaction records.


,product_id,product_name,brand,category,price,rating,material,packaging,carbon_footprint,recyclable,organic,eco_certified,sustainability_score
0,P0000,Product 0,Brand C,Personal Care,218.83,3.6,Metal,Cardboard Box,3.99,True,False,True,8.2
1,P0001,Product 1,Brand D,Food & Beverage,488.58,4.0,Recycled Plastic,Cardboard Box,3.47,False,True,False,7.8
2,P0002,Product 2,Brand E,Personal Care,491.16,4.8,Organic Cotton,Plastic Wrap,2.00,True,False,False,2.5
3,P0003,Product 3,Brand A,Electronics,447.71,3.3,Bamboo,Recycled Cardboard,2.38,False,False,True,4.4
4,P0004,Product 4,Brand C,Electronics,55.88,2.6,Organic Cotton,Recycled Cardboard,1.71,True,False,False,8.5


,user_id,product_id,interaction_score
0,U000,P0075,2
1,U000,P0150,5
2,U000,P0015,1
3,U000,P0001,5
4,U000,P0036,4


## 4. Data Exploration

Let's examine the structure and basic statistics of the generated `df_products` and `df_interactions` DataFrames.

In [ ]:
print("### Product DataFrame Info ###")
df_products.info()
print("\n### Product DataFrame Description ###")
display(df_products.describe(include='all'))
print("\n### Missing values in Product DataFrame ###")
display(df_products.isnull().sum()[df_products.isnull().sum() > 0])

print("\n### Interactions DataFrame Info ###")
df_interactions.info()
print("\n### Interactions DataFrame Description ###")
display(df_interactions.describe(include='all'))
print("\n### Missing values in Interactions DataFrame ###")
display(df_interactions.isnull().sum()[df_interactions.isnull().sum() > 0])

### Product DataFrame Info ###
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   product_id            200 non-null    object 
 1   product_name          200 non-null    object 
 2   brand                 200 non-null    object 
 3   category              200 non-null    object 
 4   price                 200 non-null    float64
 5   rating                200 non-null    float64
 6   material              200 non-null    object 
 7   packaging             200 non-null    object 
 8   carbon_footprint      200 non-null    float64
 9   recyclable            200 non-null    bool   
 10  organic               200 non-null    bool   
 11  eco_certified         200 non-null    bool   
 12  sustainability_score  200 non-null    float64
dtypes: bool(3), float64(4), object(6)
memory usage: 16.3+ KB

### Product DataFrame Description ##

,product_id,product_name,brand,category,price,rating,material,packaging,carbon_footprint,recyclable,organic,eco_certified,sustainability_score
count,200,200,200,200,200.000000,200.000000,200,200,200.000000,200,200,200,200.000000
unique,200,200,5,5,NaN,NaN,7,4,NaN,2,2,2,NaN
top,P0000,Product 0,Brand A,Home Goods,NaN,NaN,Organic Cotton,Recycled Cardboard,NaN,True,False,False,NaN
freq,1,1,46,46,NaN,NaN,34,59,NaN,135,154,172,NaN
mean,NaN,NaN,NaN,NaN,253.845050,3.807500,NaN,NaN,4.899000,NaN,NaN,NaN,5.617000
std,NaN,NaN,NaN,NaN,139.537606,0.707102,NaN,NaN,2.770206,NaN,NaN,NaN,2.620697
min,NaN,NaN,NaN,NaN,6.310000,2.500000,NaN,NaN,0.110000,NaN,NaN,NaN,1.000000
25%,NaN,NaN,NaN,NaN,146.292500,3.275000,NaN,NaN,2.632500,NaN,NaN,NaN,3.300000
50%,NaN,NaN,NaN,NaN,244.840000,3.800000,NaN,NaN,4.990000,NaN,NaN,NaN,5.700000
75%,NaN,NaN,NaN,NaN,381.720000,4.400000,NaN,NaN,7.060000,NaN,NaN,NaN,8.000000



### Missing values in Product DataFrame ###


,0



### Interactions DataFrame Info ###
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 262 entries, 0 to 261
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   user_id            262 non-null    object
 1   product_id         262 non-null    object
 2   interaction_score  262 non-null    int64 
dtypes: int64(1), object(2)
memory usage: 6.3+ KB

### Interactions DataFrame Description ###


,user_id,product_id,interaction_score
count,262,262,262.000000
unique,20,144,NaN
top,U015,P0182,NaN
freq,20,6,NaN
mean,NaN,NaN,2.893130
std,NaN,NaN,1.445044
min,NaN,NaN,1.000000
25%,NaN,NaN,2.000000
50%,NaN,NaN,3.000000
75%,NaN,NaN,4.000000



### Missing values in Interactions DataFrame ###


,0


## 5. Feature Engineering

This section prepares the product data for machine learning models. Categorical features like `category`, `brand`, `material`, and `packaging` will be encoded using `LabelEncoder`. Numerical features such as `price`, `carbon_footprint`, `rating`, and `sustainability_score` will be normalized using `MinMaxScaler`.

In [ ]:
class FeatureEngineer:
    """
    Handles feature engineering steps: encoding categorical features and normalizing numerical features.
    """
    def __init__(self):
        self.label_encoders = {}
        self.min_max_scalers = {}
        # self.encoded_cols will now store the names of the OHE columns
        self.encoded_cols = []
        self.numerical_cols_to_normalize = [
            'price', 'carbon_footprint', 'rating', 'sustainability_score'
        ]
        self.categorical_cols_to_encode = [
            'brand', 'category', 'material', 'packaging'
        ]

    def encode_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Encodes categorical features using One-Hot Encoding.
        Original categorical columns are retained in the output DataFrame for filtering/display.
        """
        df_processed_with_ohe = df.copy() # Start with a copy of the original DataFrame

        # Create one-hot encoded features from the specified categorical columns
        ohe_features_df = pd.get_dummies(df_processed_with_ohe[self.categorical_cols_to_encode],
                                        prefix=self.categorical_cols_to_encode)

        # Store the names of the generated one-hot encoded columns
        self.encoded_cols = ohe_features_df.columns.tolist()

        # Concatenate the original DataFrame (which still has original categorical columns)
        # with the new one-hot encoded features.
        # This effectively adds OHE columns without removing the originals.
        df_processed_with_ohe = pd.concat([df_processed_with_ohe, ohe_features_df], axis=1)

        # Drop any potential duplicate columns (shouldn't happen with this approach if column names are distinct)
        df_processed_with_ohe = df_processed_with_ohe.loc[:, ~df_processed_with_ohe.columns.duplicated()]

        return df_processed_with_ohe

    def normalize_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Normalizes numerical features using MinMaxScaler.
        """
        df_normalized = df.copy()
        for col in self.numerical_cols_to_normalize:
            scaler = MinMaxScaler()
            df_normalized[col + '_normalized'] = scaler.fit_transform(df_normalized[[col]])
            self.min_max_scalers[col] = scaler

        return df_normalized

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Applies both encoding and normalization to the DataFrame.
        """
        # First encode, which adds OHE columns and retains original categorical columns
        df_transformed = self.encode_features(df.copy())
        # Then normalize numerical features, adding normalized columns
        df_transformed = self.normalize_features(df_transformed)
        return df_transformed

    def save_encoders(self, path: str = 'feature_encoder.pkl'):
        """
        Saves the fitted label encoders and min-max scalers.
        """
        joblib.dump({
            'label_encoders': self.label_encoders,
            'min_max_scalers': self.min_max_scalers,
            'encoded_cols': self.encoded_cols,
            'numerical_cols_to_normalize': self.numerical_cols_to_normalize,
            'categorical_cols_to_encode': self.categorical_cols_to_encode
        }, path)
        print(f"Feature encoders and scalers saved to {path}")

    def load_encoders(self, path: str = 'feature_encoder.pkl'):
        """
        Loads the fitted label encoders and min-max scalers.
        """
        data = joblib.load(path)
        self.label_encoders = data['label_encoders']
        self.min_max_scalers = data['min_max_scalers']
        self.encoded_cols = data['encoded_cols']
        self.numerical_cols_to_normalize = data['numerical_cols_to_normalize']
        self.categorical_cols_to_encode = data['categorical_cols_to_encode']
        print(f"Feature encoders and scalers loaded from {path}")

# Initialize and apply Feature Engineering
feature_engineer = FeatureEngineer()
df_products_processed = feature_engineer.transform(df_products)

print("Product DataFrame after Feature Engineering:")
display(df_products_processed.head())
print(f"Shape of processed products DataFrame: {df_products_processed.shape}")

Product DataFrame after Feature Engineering:


,product_id,product_name,brand,category,price,rating,material,packaging,carbon_footprint,recyclable,...,material_Plastic,material_Recycled Plastic,packaging_Cardboard Box,packaging_No Packaging,packaging_Plastic Wrap,packaging_Recycled Cardboard,price_normalized,carbon_footprint_normalized,rating_normalized,sustainability_score_normalized
0,P0000,Product 0,Brand C,Personal Care,218.83,3.6,Metal,Cardboard Box,3.99,True,...,False,False,True,False,False,False,0.432778,0.402490,0.44,0.800000
1,P0001,Product 1,Brand D,Food & Beverage,488.58,4.0,Recycled Plastic,Cardboard Box,3.47,False,...,False,True,True,False,False,False,0.982100,0.348548,0.60,0.755556
2,P0002,Product 2,Brand E,Personal Care,491.16,4.8,Organic Cotton,Plastic Wrap,2.00,True,...,False,False,False,False,True,False,0.987354,0.196058,0.92,0.166667
3,P0003,Product 3,Brand A,Electronics,447.71,3.3,Bamboo,Recycled Cardboard,2.38,False,...,False,False,False,False,False,True,0.898872,0.235477,0.32,0.377778
4,P0004,Product 4,Brand C,Electronics,55.88,2.6,Organic Cotton,Recycled Cardboard,1.71,True,...,False,False,False,False,False,True,0.100945,0.165975,0.04,0.833333


Shape of processed products DataFrame: (200, 38)


## 6. Sustainability Scoring

The Eco Score will be calculated based on a weighted average of Carbon Footprint, Packaging, Recyclability, Organic Certification, Eco Certification, and the initial Sustainability Score. This score will help in ranking greener alternatives.

In [ ]:
def calculate_eco_score(df_processed: pd.DataFrame, df_original_products: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates a composite Eco Score for each product.

    Eco Score = 40% Sustainability Score + 30% Carbon (inverse) + 20% Packaging (goodness) +
                10% Recyclability + 10% Organic + 10% Eco-certified + 10% Rating (normalized)

    Args:
        df_processed (pd.DataFrame): The DataFrame containing product data with normalized and encoded features.
        df_original_products (pd.DataFrame): The original DataFrame containing product data, specifically for 'packaging'.

    Returns:
        pd.DataFrame: The DataFrame with an added 'eco_score' column.
    """
    df_copy = df_processed.copy()

    # Ensure required normalized columns exist, if not, create them (should be created by FeatureEngineer)
    for col_suffix in ['sustainability_score', 'carbon_footprint', 'rating']:
        normalized_col = f'{col_suffix}_normalized'
        if normalized_col not in df_copy.columns:
            print(f"Warning: '{normalized_col}' not found. Please ensure feature engineering is applied.")
            scaler = MinMaxScaler()
            df_copy[normalized_col] = scaler.fit_transform(df_processed[[col_suffix]])

    # Invert carbon footprint so lower values mean higher score
    df_copy['carbon_footprint_inverted_normalized'] = 1 - df_copy['carbon_footprint_normalized']

    # Convert boolean to numeric (True=1, False=0) - these columns still exist in df_processed
    df_copy['recyclable_num'] = df_copy['recyclable'].astype(int)
    df_copy['organic_num'] = df_copy['organic'].astype(int)
    df_copy['eco_certified_num'] = df_copy['eco_certified'].astype(int)

    # Assign packaging goodness score (example scoring)
    packaging_score_map = {
        'No Packaging': 1.0,
        'Recycled Cardboard': 0.8,
        'Cardboard Box': 0.6,
        'Plastic Wrap': 0.2
    }

    # Calculate packaging_goodness_score from original dataframe and align it
    # Temporarily set product_id as index for df_copy to align with packaging_scores series
    df_copy = df_copy.set_index('product_id')
    packaging_scores_series = df_original_products.set_index('product_id')['packaging'].map(packaging_score_map)
    df_copy['packaging_goodness_score'] = packaging_scores_series
    df_copy = df_copy.reset_index() # Reset product_id back to a column

    # Define weights for Eco Score components
    weights = {
        'sustainability_score_normalized': 0.40,
        'carbon_footprint_inverted_normalized': 0.30,
        'packaging_goodness_score': 0.20,
        'rating_normalized': 0.10,
    }

    # Calculate initial composite score
    df_copy['eco_score_composite'] = (
        df_copy['sustainability_score_normalized'] * weights['sustainability_score_normalized'] +
        df_copy['carbon_footprint_inverted_normalized'] * weights['carbon_footprint_inverted_normalized'] +
        df_copy['packaging_goodness_score'] * weights['packaging_goodness_score'] +
        df_copy['rating_normalized'] * weights['rating_normalized']
    )

    # Normalize eco_score_composite to a 0-1 range for internal calculations in the recommendation engine
    scaler_0_1 = MinMaxScaler(feature_range=(0, 1))
    df_copy['eco_score_normalized'] = scaler_0_1.fit_transform(df_copy[['eco_score_composite']])

    # Scale eco_score_composite to a 1-10 range for better interpretability and display
    scaler_1_10 = MinMaxScaler(feature_range=(1, 10))
    df_copy['eco_score'] = scaler_1_10.fit_transform(df_copy[['eco_score_composite']])

    # Drop the intermediate composite score
    df_copy = df_copy.drop(columns=['eco_score_composite'])

    return df_copy

# Call the function with both processed and original dataframes
df_products_processed = calculate_eco_score(df_products_processed, df_products)

print("Product DataFrame with calculated Eco Score:")
display(df_products_processed[['product_id', 'product_name', 'sustainability_score', 'carbon_footprint', 'packaging_goodness_score', 'rating', 'eco_score', 'eco_score_normalized']].head())

Product DataFrame with calculated Eco Score:


,product_id,product_name,sustainability_score,carbon_footprint,packaging_goodness_score,rating,eco_score,eco_score_normalized
0,P0000,Product 0,8.2,3.99,0.6,3.6,6.744460,0.638273
1,P0001,Product 1,7.8,3.47,0.6,4.0,6.904409,0.656045
2,P0002,Product 2,2.5,2.00,0.2,4.8,4.263803,0.362645
3,P0003,Product 3,4.4,2.38,0.8,3.3,5.736389,0.526265
4,P0004,Product 4,8.5,1.71,0.8,2.6,7.680383,0.742265


## 7. Train Recommendation Engine

This section implements the `AIRecommendationEngine` class, which will encapsulate the entire recommendation logic. It will include methods for loading data, preprocessing, training various recommendation models (Content-Based and Collaborative Filtering), and combining them into a hybrid system.

In [ ]:
class AIRecommendationEngine:
    """
    An AI Recommendation Engine for sustainable products, combining content-based,
    collaborative filtering, and hybrid approaches.
    """
    def __init__(self, feature_engineer: FeatureEngineer):
        self.df_products: Optional[pd.DataFrame] = None
        self.df_interactions: Optional[pd.DataFrame] = None
        self.df_products_processed: Optional[pd.DataFrame] = None
        self.feature_engineer = feature_engineer
        self.content_similarity_matrix: Optional[np.ndarray] = None
        self.user_product_matrix: Optional[pd.DataFrame] = None
        self.product_to_idx: Optional[Dict[str, int]] = None
        self.idx_to_product: Optional[Dict[int, str]] = None
        self.user_to_idx: Optional[Dict[str, int]] = None
        self.idx_to_user: Optional[Dict[int, str]] = None
        self.model: Optional[Any] = None # Placeholder for potential future model

    def load_dataset(self, df_products: pd.DataFrame, df_interactions: pd.DataFrame):
        """
        Loads product and interaction datasets.
        """
        self.df_products = df_products
        self.df_interactions = df_interactions
        print("Datasets loaded successfully.")

    def preprocess(self):
        """
        Applies feature engineering and calculates Eco Score.
        """
        if self.df_products is None:
            raise ValueError("Product DataFrame not loaded. Call load_dataset first.")

        # Apply feature engineering
        self.df_products_processed = self.feature_engineer.transform(self.df_products)

        # Calculate Eco Score
        self.df_products_processed = calculate_eco_score(self.df_products_processed, self.df_products)

        # Prepare for content-based: select features for similarity
        content_features = []
        # Add normalized numerical features
        content_features.extend([col for col in self.df_products_processed.columns if '_normalized' in col])
        # Add one-hot encoded categorical features (using the list stored by FeatureEngineer)
        content_features.extend(self.feature_engineer.encoded_cols)
        # Add eco_score_normalized if it exists and is not already included
        if 'eco_score_normalized' in self.df_products_processed.columns and 'eco_score_normalized' not in content_features:
            content_features.append('eco_score_normalized')

        # Remove duplicates and ensure all columns exist before selection
        content_features = list(set(content_features))
        # Filter to only columns that actually exist in the DataFrame
        content_features = [col for col in content_features if col in self.df_products_processed.columns]

        self.content_features_df = self.df_products_processed[content_features]

        # Create mappings for product and user IDs
        self.product_to_idx = {pid: i for i, pid in enumerate(self.df_products_processed['product_id'])}
        self.idx_to_product = {i: pid for pid, i in self.product_to_idx.items()}

        if self.df_interactions is not None:
            unique_users = self.df_interactions['user_id'].unique()
            self.user_to_idx = {uid: i for i, uid in enumerate(unique_users)}
            self.idx_to_user = {i: uid for uid, i in self.user_to_idx.items()}

        print("Data preprocessing complete.")
        print(f"Content features for similarity: {self.content_features_df.shape}")

    def train_content_based(self):
        """
        Trains the content-based recommendation model using cosine similarity.
        """
        if self.content_features_df is None:
            raise ValueError("Content features not prepared. Call preprocess first.")

        # Calculate cosine similarity between products based on their features
        self.content_similarity_matrix = cosine_similarity(self.content_features_df)
        print("Content-based model trained. Content similarity matrix generated.")
        print(f"Content similarity matrix shape: {self.content_similarity_matrix.shape}")

    def train_collaborative_filtering(self):
        """
        Trains the collaborative filtering model using user-product interactions.
        Creates a user-product interaction matrix.
        """
        if self.df_interactions is None or self.product_to_idx is None or self.user_to_idx is None:
            raise ValueError("Interactions data or mappings not prepared. Call load_dataset and preprocess first.")

        # Create a user-product matrix for collaborative filtering
        # Fill missing values with 0 (assuming no interaction means 0 rating or no preference)
        user_product_pivot = self.df_interactions.pivot_table(
            index='user_id', columns='product_id', values='interaction_score'
        ).fillna(0)

        # Align with the product_to_idx order for consistency with other matrices
        # Get all product IDs, including those not in interactions, and fill with 0 if not present
        all_product_ids = [self.idx_to_product[i] for i in range(len(self.product_to_idx))]
        self.user_product_matrix = user_product_pivot.reindex(columns=all_product_ids, fill_value=0)
        self.user_product_matrix = self.user_product_matrix.reindex(index=[self.idx_to_user[i] for i in range(len(self.user_to_idx))], fill_value=0)

        print("Collaborative filtering model trained. User-product matrix created.")
        print(f"User-product matrix shape: {self.user_product_matrix.shape}")

    def train(self):
        """
        Orchestrates the training of all recommendation models.
        """
        self.preprocess()
        self.train_content_based()
        self.train_collaborative_filtering()
        print("All recommendation models trained successfully.")

    def recommend_products(
        self,
        user_id: str,
        num_recommendations: int = 10,
        content_weight: float = 0.5,
        cf_weight: float = 0.5,
        user_preferences: Optional[Dict[str, Any]] = None
    ) -> pd.DataFrame:
        """
        Generates hybrid product recommendations for a given user.

        Args:
            user_id (str): The ID of the user for whom to generate recommendations.
            num_recommendations (int): The number of products to recommend.
            content_weight (float): Weight for content-based similarity in hybrid scoring.
            cf_weight (float): Weight for collaborative filtering score in hybrid scoring.
            user_preferences (Optional[Dict[str, Any]]): Dictionary of user preferences
                                                           (e.g., {'category': 'Apparel', 'max_price': 100}).

        Returns:
            pd.DataFrame: A DataFrame of recommended products.
        """
        if self.df_products_processed is None or self.content_similarity_matrix is None or \
           self.user_product_matrix is None or self.product_to_idx is None or self.user_to_idx is None:
            raise ValueError("Engine not trained. Call train() first.")

        if user_id not in self.user_to_idx:
            print(f"User '{user_id}' not found in interactions data. Reverting to popular/content-based recommendations.")
            # Fallback for new users: use content-based or popular products
            # For simplicity, we'll just recommend top eco-score products if user not found
            return self.df_products_processed.sort_values(by='eco_score', ascending=False).head(num_recommendations)

        user_idx = self.user_to_idx[user_id]
        user_interactions = self.user_product_matrix.loc[user_id]
        interacted_product_indices = user_interactions[user_interactions > 0].index.map(self.product_to_idx).tolist()

        # --- Content-Based Recommendations ---
        # If user has interacted with items, find similar items based on content
        content_scores = np.zeros(len(self.df_products_processed))
        if interacted_product_indices:
            # Average similarity of interacted products to all other products
            for p_idx in interacted_product_indices:
                content_scores += self.content_similarity_matrix[p_idx]
            content_scores /= len(interacted_product_indices)
        else:
            # If no interactions, default to general content similarity or popular items (eco-score)
            content_scores = self.df_products_processed['eco_score_normalized'].values # Using eco_score as a fallback content score

        # --- Collaborative Filtering Recommendations ---
        # Calculate user-user similarity and then predict ratings
        # For simplicity, let's use item-item collaborative filtering approach here based on known interactions
        # A more robust user-user CF would involve calculating user similarity directly from user_product_matrix

        # Predict ratings for unrated items using item-item collaborative filtering
        cf_scores = np.zeros(len(self.df_products_processed))
        if interacted_product_indices:
            for i in range(len(self.df_products_processed)): # for each target product 'i'
                if i not in interacted_product_indices: # only consider products the user hasn't interacted with
                    # Weighted sum of similarities to already interacted items, multiplied by user's rating for those items
                    weighted_sum = 0
                    sum_of_similarities = 0
                    for j in interacted_product_indices:
                        similarity = self.content_similarity_matrix[i, j] # Using content sim for item-item for now
                        # Use actual interaction score from user_interactions for product j
                        interaction_score = user_interactions.iloc[self.product_to_idx[self.idx_to_product[j]]] # Original line had an issue with index here if the interaction was for a column not explicitly in user_interactions, user_interactions.loc[self.idx_to_product[j]] would be better, but the original was user_interactions.iloc[idx], which worked for mapped index. This will use the product_id to get the score.
                        weighted_sum += similarity * interaction_score
                        sum_of_similarities += abs(similarity)
                    if sum_of_similarities > 0:
                        cf_scores[i] = weighted_sum / sum_of_similarities

            # Normalize CF scores to a 0-1 range for consistent weighting
            if np.max(cf_scores) > 0:
                cf_scores = MinMaxScaler().fit_transform(cf_scores.reshape(-1, 1)).flatten()
        else:
            # If no interactions, CF scores are all 0
            cf_scores = np.zeros(len(self.df_products_processed))

        # --- Hybrid Scoring ---
        # Combine content-based and collaborative filtering scores
        hybrid_scores = (content_weight * content_scores) + (cf_weight * cf_scores)

        # Add Eco Score and Popularity (e.g., average rating) to influence final recommendations
        # Ensure eco_score is normalized for combination
        eco_scores_norm = self.df_products_processed['eco_score_normalized'].values
        # Popularity score (e.g., average rating)
        popularity_scores_norm = self.df_products_processed['rating_normalized'].values

        # Adjust hybrid score with sustainability and popularity
        # More weight to eco_score and popularity
        hybrid_scores = hybrid_scores + (0.3 * eco_scores_norm) + (0.1 * popularity_scores_norm)

        # Filter out already interacted products
        recommended_product_indices = [i for i in range(len(self.df_products_processed)) if i not in interacted_product_indices]
        final_scores = pd.Series(hybrid_scores, index=self.df_products_processed.index).iloc[recommended_product_indices]

        # Apply user preferences as filters or additional score modifiers
        if user_preferences:
            print(f"DEBUG: Columns in self.df_products_processed before preference filtering: {self.df_products_processed.columns.tolist()}")
            print(f"DEBUG: 'category' in self.df_products_processed.columns: {'category' in self.df_products_processed.columns}")
            print(f"DEBUG: Products before preference filtering: {len(final_scores)} products")

            # Example: Filter by category
            if 'category' in user_preferences:
                target_category = user_preferences['category']
                # Now 'category' column exists in self.df_products_processed for filtering
                category_filter = self.df_products_processed.loc[final_scores.index, 'category'] == target_category
                final_scores = final_scores[category_filter]
                print(f"DEBUG: Products after category filter ('{target_category}'): {len(final_scores)} products")

            # Example: Filter by max budget
            if 'max_price' in user_preferences:
                max_price = user_preferences['max_price']
                price_filter = self.df_products_processed.loc[final_scores.index, 'price'] <= max_price
                final_scores = final_scores[price_filter]
                print(f"DEBUG: Products after max_price filter ('{max_price}'): {len(final_scores)} products")

            # Example: Organic only
            if user_preferences.get('organic_only', False):
                organic_filter = self.df_products_processed.loc[final_scores.index, 'organic'] == True
                final_scores = final_scores[organic_filter]
                print(f"DEBUG: Products after organic_only filter: {len(final_scores)} products")

            # Example: Recyclable only
            if user_preferences.get('recyclable_only', False):
                recyclable_filter = self.df_products_processed.loc[final_scores.index, 'recyclable'] == True
                final_scores = final_scores[recyclable_filter]
                print(f"DEBUG: Products after recyclable_only filter: {len(final_scores)} products")

            # Example: Max Carbon Footprint
            if 'max_carbon_footprint' in user_preferences:
                max_cf = user_preferences['max_carbon_footprint']
                cf_filter = self.df_products_processed.loc[final_scores.index, 'carbon_footprint'] <= max_cf
                final_scores = final_scores[cf_filter]
                print(f"DEBUG: Products after max_carbon_footprint filter ('{max_cf}'): {len(final_scores)} products")

        # Sort by score and get top recommendations
        top_n_recommendations = final_scores.nlargest(num_recommendations)

        recommended_df = self.df_products_processed.loc[top_n_recommendations.index].copy()
        recommended_df['recommendation_score'] = top_n_recommendations.values
        recommended_df['rank'] = range(1, len(recommended_df) + 1)

        # Re-add original 'category' and other potentially dropped human-readable columns for display
        # The original categorical columns are removed by pd.get_dummies during preprocessing.
        # We need to retrieve them from the original df_products.
        if self.df_products is not None:
            # Columns from the original df_products that we want to ensure are in the final output
            cols_to_readd_from_original = ['product_name', 'category', 'brand', 'material', 'packaging']

            # Filter to only columns actually present in self.df_products and not already in recommended_df
            cols_to_readd_from_original = [
                col for col in cols_to_readd_from_original
                if col in self.df_products.columns and col not in recommended_df.columns
            ]

            if cols_to_readd_from_original:
                # Create a DataFrame from self.df_products with only product_id and the needed columns
                original_info_for_merge = self.df_products[['product_id'] + cols_to_readd_from_original].set_index('product_id')

                # Join recommended_df (indexed by product_id) with the selected original columns
                recommended_df = recommended_df.set_index('product_id').join(
                    original_info_for_merge, how='left'
                )
                recommended_df = recommended_df.reset_index()

        return recommended_df

    def recommend_eco_alternatives(
        self,
        product_id: str,
        num_alternatives: int = 5,
        eco_score_weight: float = 0.6,
        content_similarity_weight: float = 0.4
    ) -> pd.DataFrame:
        """
        Recommends greener alternatives for a given product.

        Args:
            product_id (str): The ID of the product for which to find alternatives.
            num_alternatives (int): The number of greener alternatives to recommend.
            eco_score_weight (float): Weight for eco_score in ranking alternatives.
            content_similarity_weight (float): Weight for content similarity in ranking alternatives.

        Returns:
            pd.DataFrame: A DataFrame of recommended greener alternative products.
        """
        if self.df_products_processed is None or self.content_similarity_matrix is None:
            raise ValueError("Engine not trained or product data not processed. Call train() first.")

        # Ensure df_products_processed has product_id as its index for consistent lookups
        df_products_indexed = self.df_products_processed.set_index('product_id')

        if product_id not in df_products_indexed.index:
            raise ValueError(f"Product ID '{product_id}' not found.")

        product_idx_int = self.product_to_idx[product_id] # Get integer index for content_similarity_matrix
        current_product_eco_score = df_products_indexed.loc[product_id, 'eco_score_normalized']

        # Get content similarities with all other products, indexed by product_id
        product_similarities = pd.Series(
            self.content_similarity_matrix[product_idx_int],
            index=self.df_products_processed['product_id'] # Use product_id column for index
        )

        # Exclude the product itself from alternatives
        product_similarities = product_similarities.drop(product_id)

        # Create a DataFrame for potential alternatives
        # Ensure alignment by product_id
        alternative_data = df_products_indexed[[
            'eco_score_normalized'
        ]].copy()
        alternative_data['content_similarity'] = product_similarities.reindex(alternative_data.index, fill_value=0)

        # Filter for products with higher eco_score than the current product
        greener_alternatives = alternative_data[
            alternative_data['eco_score_normalized'] > current_product_eco_score
        ].copy() # Explicit copy to avoid SettingWithCopyWarning

        if greener_alternatives.empty:
            print(f"No greener alternatives found for product '{product_id}'.")
            return pd.DataFrame()

        # Combine eco_score and content_similarity for ranking
        greener_alternatives.loc[:, 'alternative_score'] = ( # Use .loc for assignment to avoid SettingWithCopyWarning
            greener_alternatives['eco_score_normalized'] * eco_score_weight +
            greener_alternatives['content_similarity'] * content_similarity_weight
        )

        # Sort by the combined alternative score and get top N
        top_alternatives = greener_alternatives.sort_values(
            by='alternative_score', ascending=False
        ).head(num_alternatives)

        # Retrieve the full product details for these top alternatives
        # Since df_products_indexed is already indexed by product_id, direct .loc works
        recommended_df = df_products_indexed.loc[top_alternatives.index].copy()

        # Add the 'alternative_score' and 'rank'
        recommended_df['alternative_score'] = top_alternatives['alternative_score']
        recommended_df['rank'] = range(1, len(recommended_df) + 1)
        recommended_df = recommended_df.reset_index() # Reset product_id from index to column for final output

        # Re-add original human-readable columns for display
        if self.df_products is not None:
            # The original df_products contains 'product_id' as a column and other original columns
            # We want to ensure 'product_name', 'category', 'brand', etc. are present if they were not part of the processed features
            cols_to_readd_from_original = ['product_name', 'category', 'brand', 'material', 'packaging']

            # Filter to only columns actually present in self.df_products and not already in recommended_df
            cols_to_readd_from_original = [
                col for col in cols_to_readd_from_original
                if col in self.df_products.columns and col not in recommended_df.columns
            ]

            if cols_to_readd_from_original:
                # Select only relevant columns from original df_products and set product_id as index
                original_info_for_merge = self.df_products[['product_id'] + cols_to_readd_from_original].set_index('product_id')

                # Join with recommended_df which has product_id as a column (after reset_index)
                recommended_df = recommended_df.set_index('product_id').join(
                    original_info_for_merge, how='left'
                )
                recommended_df = recommended_df.reset_index()

        return recommended_df.sort_values(by='alternative_score', ascending=False)

In [ ]:
# Initialize and train the recommendation engine
engine = AIRecommendationEngine(feature_engineer=feature_engineer)
engine.load_dataset(df_products, df_interactions)
engine.train()

print("\n--- Engine State After Training ---")
print(f"Content Similarity Matrix (first 5x5):\n{engine.content_similarity_matrix[:5, :5]}")
print(f"\nUser-Product Interaction Matrix (first 5x5):\n{engine.user_product_matrix.iloc[:5, :5]}")

Datasets loaded successfully.
Data preprocessing complete.
Content features for similarity: (200, 27)
Content-based model trained. Content similarity matrix generated.
Content similarity matrix shape: (200, 200)
Collaborative filtering model trained. User-product matrix created.
User-product matrix shape: (20, 200)
All recommendation models trained successfully.

--- Engine State After Training ---
Content Similarity Matrix (first 5x5):
[[1.         0.50709399 0.4377609  0.28857661 0.46394307]
 [0.50709399 1.         0.36608933 0.35684662 0.28716916]
 [0.4377609  0.36608933 1.         0.33235382 0.355994  ]
 [0.28857661 0.35684662 0.33235382 1.         0.58336421]
 [0.46394307 0.28716916 0.355994   0.58336421 1.        ]]

User-Product Interaction Matrix (first 5x5):
product_id  P0000  P0001  P0002  P0003  P0004
user_id                                      
U000            0    5.0    0.0      0    0.0
U001            0    0.0    0.0      0    0.0
U002            0    1.0    4.0      0

### 9.1. Recommend Eco-Friendly Alternatives

Let's test the `recommend_eco_alternatives` method by selecting a product and finding greener options.

## 9. Recommend Products

Now, let's use the trained `AIRecommendationEngine` to recommend products for a specific user, demonstrating the hybrid recommendation approach and the application of user preferences.

In [ ]:
# Example: Recommend products for a specific user (e.g., U001)
user_id_to_recommend = 'U001'
num_recommendations_display = 10

print(f"\n--- Recommendations for User: {user_id_to_recommend} ---")
recommended_products = engine.recommend_products(
    user_id=user_id_to_recommend,
    num_recommendations=num_recommendations_display,
    content_weight=0.6, # Adjust weights for hybrid model
    cf_weight=0.4
)

if not recommended_products.empty:
    display(recommended_products[[
        'product_name', 'category', 'price', 'rating', 'carbon_footprint',
        'eco_score', 'recommendation_score', 'rank'
    ]])
else:
    print(f"No recommendations found for user {user_id_to_recommend} with the given parameters.")

# Example with user preferences (e.g., organic products within a budget in a specific category)
print(f"\n--- Recommendations for User: {user_id_to_recommend} with Preferences (Relaxed) ---")
user_specific_preferences = {
    'category': 'Apparel',
    'max_price': 300, # Relaxed max_price
    # 'organic_only': True, # Removed organic_only filter for demonstration
    'max_carbon_footprint': 3.0
}

recommended_products_with_prefs = engine.recommend_products(
    user_id=user_id_to_recommend,
    num_recommendations=num_recommendations_display,
    content_weight=0.5,
    cf_weight=0.5,
    user_preferences=user_specific_preferences
)

if not recommended_products_with_prefs.empty:
    print(f"Preferences: {user_specific_preferences}")
    display(recommended_products_with_prefs[[
        'product_name', 'category', 'price', 'rating', 'carbon_footprint',
        'eco_score', 'recommendation_score', 'rank'
    ]])
else:
    print(f"No recommendations found for user {user_id_to_recommend} with the specified preferences: {user_specific_preferences}")



--- Recommendations for User: U001 ---


,product_name,category,price,rating,carbon_footprint,eco_score,recommendation_score,rank
42,Product 42,Apparel,216.44,5.0,1.02,9.379599,0.986699,1
7,Product 7,Food & Beverage,162.63,4.7,0.41,10.000000,0.984137,2
31,Product 31,Personal Care,285.04,4.7,2.86,9.301436,0.982966,3
158,Product 158,Home Goods,188.71,4.8,3.41,8.218132,0.963595,4
91,Product 91,Personal Care,463.46,4.9,5.01,7.808357,0.952879,5
123,Product 123,Personal Care,383.24,5.0,4.60,8.191854,0.950173,6
160,Product 160,Home Goods,112.39,4.5,1.00,8.572223,0.941708,7
145,Product 145,Food & Beverage,238.81,4.9,2.16,7.411375,0.931575,8
184,Product 184,Apparel,193.07,5.0,2.90,7.693586,0.927726,9
169,Product 169,Electronics,400.63,5.0,5.08,7.779233,0.923721,10



--- Recommendations for User: U001 with Preferences (Relaxed) ---
DEBUG: Columns in self.df_products_processed before preference filtering: ['product_id', 'product_name', 'brand', 'category', 'price', 'rating', 'material', 'packaging', 'carbon_footprint', 'recyclable', 'organic', 'eco_certified', 'sustainability_score', 'brand_Brand A', 'brand_Brand B', 'brand_Brand C', 'brand_Brand D', 'brand_Brand E', 'category_Apparel', 'category_Electronics', 'category_Food & Beverage', 'category_Home Goods', 'category_Personal Care', 'material_Bamboo', 'material_Cotton', 'material_Glass', 'material_Metal', 'material_Organic Cotton', 'material_Plastic', 'material_Recycled Plastic', 'packaging_Cardboard Box', 'packaging_No Packaging', 'packaging_Plastic Wrap', 'packaging_Recycled Cardboard', 'price_normalized', 'carbon_footprint_normalized', 'rating_normalized', 'sustainability_score_normalized', 'carbon_footprint_inverted_normalized', 'recyclable_num', 'organic_num', 'eco_certified_num', 'packagin

,product_name,category,price,rating,carbon_footprint,eco_score,recommendation_score,rank
42,Product 42,Apparel,216.44,5.0,1.02,9.379599,1.020620,1
184,Product 184,Apparel,193.07,5.0,2.90,7.693586,0.966788,2
12,Product 12,Apparel,253.40,3.5,2.41,8.775897,0.946872,3
26,Product 26,Apparel,213.45,3.8,1.99,7.968563,0.891291,4
149,Product 149,Apparel,149.23,3.1,0.16,6.019890,0.765596,5
82,Product 82,Apparel,213.59,3.1,1.23,5.600793,0.751660,6


## 8. Product Search (Semantic Search with Sentence Transformers and FAISS)

This section implements a semantic search functionality, allowing users to find products based on natural language queries. It uses `SentenceTransformer` to convert product descriptions into embeddings and `FAISS` for efficient similarity search.

In [ ]:
class SemanticSearchEngine:
    """
    Implements semantic search for products using Sentence Transformers and FAISS.
    """
    def __init__(self, df_products: pd.DataFrame):
        self.df_products = df_products
        self.model = SentenceTransformer('all-MiniLM-L6-v2') # Smaller model for Colab
        self.product_descriptions = df_products['product_name'] + ' ' + df_products['category'] + ' ' + df_products['brand'] + ' ' + df_products['material'] + ' ' + df_products['packaging'].astype(str)
        self.embeddings: Optional[np.ndarray] = None
        self.index: Optional[faiss.IndexFlatL2] = None
        print("SemanticSearchEngine initialized with 'all-MiniLM-L6-v2' model.")

    def build_index(self):
        """
        Generates embeddings for all product descriptions and builds a FAISS index.
        """
        print("Generating embeddings for product descriptions...")
        self.embeddings = self.model.encode(self.product_descriptions.tolist(), show_progress_bar=True)

        # FAISS index (L2 distance for cosine similarity on normalized vectors)
        dimension = self.embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dimension)
        faiss.normalize_L2(self.embeddings)
        self.index.add(self.embeddings) # Add the product embeddings to the index
        print(f"FAISS index built with {self.index.ntotal} vectors of dimension {dimension}.")

    def search(self, query: str, k: int = 5) -> pd.DataFrame:
        """
        Performs a semantic search for products based on a query.

        Args:
            query (str): The search query.
            k (int): The number of top similar products to return.

        Returns:
            pd.DataFrame: A DataFrame of the top k most semantically similar products.
        """
        query_embedding = self.model.encode([query])
        faiss.normalize_L2(query_embedding)

        distances, indices = self.index.search(query_embedding, k) # Search the index

        # Retrieve the actual product data using the indices
        results = self.df_products.iloc[indices[0]].copy()
        results['search_score'] = 1 - distances[0] # Convert L2 distance to a similarity score (0 to 1)

        return results.sort_values(by='search_score', ascending=False)

# Initialize and build the semantic search engine
semantic_engine = SemanticSearchEngine(df_products)
semantic_engine.build_index()

# Example Search
print("\n--- Example Semantic Search ---")
query_product = "eco-friendly water bottle"
search_results = semantic_engine.search(query_product, k=5)
print(f"Top 5 products for query: '{query_product}'")
display(search_results[['product_name', 'category', 'brand', 'carbon_footprint', 'eco_certified', 'search_score']])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SemanticSearchEngine initialized with 'all-MiniLM-L6-v2' model.
Generating embeddings for product descriptions...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

FAISS index built with 200 vectors of dimension 384.

--- Example Semantic Search ---
Top 5 products for query: 'eco-friendly water bottle'


,product_name,category,brand,carbon_footprint,eco_certified,search_score
166,Product 166,Food & Beverage,Brand A,0.67,False,-0.173137
75,Product 75,Food & Beverage,Brand E,7.04,False,-0.208644
7,Product 7,Food & Beverage,Brand A,0.41,True,-0.209295
128,Product 128,Food & Beverage,Brand A,9.37,False,-0.221032
151,Product 151,Food & Beverage,Brand A,8.64,False,-0.234045
